Cache Lookup Logic

In [ ]:
1+1

In [ ]:
import sqlite3

conn = sqlite3.connect("geo_cache.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS geo_cache (
    address_hash TEXT PRIMARY KEY,
    normalized_address TEXT,
    city TEXT,
    state TEXT,
    pincode TEXT,
    latitude REAL,
    longitude REAL,
    confidence REAL,
    source TEXT,
    created_at TEXT DEFAULT CURRENT_TIMESTAMP
);
""")

conn.commit()


In [11]:
import hashlib

def generate_address_hash(components: dict) -> str:
    key_parts = [
        components.get("house_number"),
        components.get("street"),
        components.get("landmark"),
        components.get("village"),
        components.get("taluk"),
        components.get("city"),
        components.get("district"),
        components.get("state"),
        components.get("pincode")
    ]
    key = "|".join([str(p).lower() for p in key_parts if p])
    return hashlib.sha256(key.encode()).hexdigest()


In [16]:
sample_records = [
    {
    "normalized_address": "Flat 302, Prestige Sunrise, Whitefield, Bengaluru, Karnataka 560066",
    "components": {
        "house_number": "302",
        "street": None,
        "landmark": "Prestige Sunrise",
        "village": None,
        "taluk": "Whitefield",
        "city": "Bengaluru",
        "district": "Bengaluru Urban",
        "state": "Karnataka",
        "pincode": "560066"
    },
    "latitude": 12.9698,
    "longitude": 77.7499,
    "confidence": 0.93,
    "source": "cache"
    },
    {
    "normalized_address": "Near Government School, Yelahanka, Bengaluru, Karnataka 560064",
    "components": {
        "house_number": None,
        "street": None,
        "landmark": "Government School",
        "village": None,
        "taluk": "Yelahanka",
        "city": "Bengaluru",
        "district": "Bengaluru Urban",
        "state": "Karnataka",
        "pincode": "560064"
    },
    "latitude": 13.1016,
    "longitude": 77.5963,
    "confidence": 0.85,
    "source": "cache"
    },
    {
    "normalized_address": "Opposite Temple, K R Puram Village, Tumkur District, Karnataka",
    "components": {
        "house_number": None,
        "street": None,
        "landmark": "Temple",
        "village": "K R Puram",
        "taluk": None,
        "city": None,
        "district": "Tumkur",
        "state": "Karnataka",
        "pincode": None
    },
    "latitude": 13.3409,
    "longitude": 77.1010,
    "confidence": 0.78,
    "source": "cache"
    }
]

In [17]:
for i in sample_records:
    
    print(i)

{'normalized_address': 'Flat 302, Prestige Sunrise, Whitefield, Bengaluru, Karnataka 560066', 'components': {'house_number': '302', 'street': None, 'landmark': 'Prestige Sunrise', 'village': None, 'taluk': 'Whitefield', 'city': 'Bengaluru', 'district': 'Bengaluru Urban', 'state': 'Karnataka', 'pincode': '560066'}, 'latitude': 12.9698, 'longitude': 77.7499, 'confidence': 0.93, 'source': 'cache'}
{'normalized_address': 'Near Government School, Yelahanka, Bengaluru, Karnataka 560064', 'components': {'house_number': None, 'street': None, 'landmark': 'Government School', 'village': None, 'taluk': 'Yelahanka', 'city': 'Bengaluru', 'district': 'Bengaluru Urban', 'state': 'Karnataka', 'pincode': '560064'}, 'latitude': 13.1016, 'longitude': 77.5963, 'confidence': 0.85, 'source': 'cache'}
{'normalized_address': 'Opposite Temple, K R Puram Village, Tumkur District, Karnataka', 'components': {'house_number': None, 'street': None, 'landmark': 'Temple', 'village': 'K R Puram', 'taluk': None, 'city':

In [18]:
for sample_record in sample_records:
    sample_record["address_hash"] = generate_address_hash(
        sample_record["components"]
    )


In [19]:
def insert_cache(conn, record: dict):
    cursor = conn.cursor()
    cursor.execute("""
        INSERT OR REPLACE INTO geo_cache
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
    """, (
        record["address_hash"],
        record["normalized_address"],
        record["city"],
        record["state"],
        record["pincode"],
        record["latitude"],
        record["longitude"],
        record["confidence"],
        record["source"]
    ))
    conn.commit()

for sample_record in sample_records:
    insert_cache(conn, {
        "address_hash": sample_record["address_hash"],
        "normalized_address": sample_record["normalized_address"],
        "city": sample_record["components"]["city"],
        "state": sample_record["components"]["state"],
        "pincode": sample_record["components"]["pincode"],
        "latitude": sample_record["latitude"],
        "longitude": sample_record["longitude"],
        "confidence": sample_record["confidence"],
        "source": sample_record["source"]
    })

In [20]:
#Verify insertion (quick check)
cursor = conn.cursor()
cursor.execute("SELECT * FROM geo_cache")
rows = cursor.fetchall()

for row in rows:
    print(row)


('252c37715291bed8fd341d1aaf6c8b9c5edf718fd1a23b58d17570702c6eef50', 'Flat 302, Prestige Sunrise, Whitefield, Bengaluru, Karnataka 560066', 'Bengaluru', 'Karnataka', '560066', 12.9698, 77.7499, 0.93, 'cache', '2026-01-08 19:51:15')
('7faf402bc37828fcff9703bbf9431e046b4a62509b9c0e31a296eb019134b2cf', 'Near Government School, Yelahanka, Bengaluru, Karnataka 560064', 'Bengaluru', 'Karnataka', '560064', 13.1016, 77.5963, 0.85, 'cache', '2026-01-08 19:51:15')
('1f548d9e72de18e3aab447216e6c46a2352270b06cbef66088c39065f4eb7986', 'Opposite Temple, K R Puram Village, Tumkur District, Karnataka', None, 'Karnataka', None, 13.3409, 77.101, 0.78, 'cache', '2026-01-08 19:51:15')


In [21]:
def lookup_cache(conn, address_hash: str, min_confidence=0.75):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT latitude, longitude, confidence, source
        FROM geo_cache
        WHERE address_hash = ?
    """, (address_hash,))
    
    row = cursor.fetchone()
    
    if row and row[2] >= min_confidence:
        return {
            "latitude": row[0],
            "longitude": row[1],
            "confidence": row[2],
            "source": "cache",
            "decision": "accepted"
        }
    
    return None


In [27]:
parsed_not_found = {
  "components": {
    "house_number": None,
    "street": None,
    "landmark": None,
    "village": None,
    "taluk": None,
    "city": "Bengaluru",
    "district": None,
    "state": "Karnataka",
    "pincode": 560066
  },
  "normalized_address": "Flat302, Prestige Sunrise, Whitefield, Bengaluru, Karnataka, 560066",
  "confidence": 0.8
}
parsed_found = {
  "components": {
      "house_number": "302",
      "street": None,
      "landmark": "Prestige Sunrise",
      "village": None,
      "taluk": "Whitefield",
      "city": "Bengaluru",
      "district": "Bengaluru Urban",
      "state": "Karnataka",
      "pincode": "560066"
  },
  "normalized_address": "Flat 302, Prestige Sunrise, Whitefield, Bengaluru, Karnataka 560066",
  "confidence": 0.8
}

In [ ]:
# parsed = parse_address(raw_address)
address_hash = generate_address_hash(parsed_found["components"])
cache_result = lookup_cache(conn, address_hash)

if cache_result:
    print("CACHE HIT", cache_result)
else:
    print("CACHE MISS → External resolution required")


CACHE MISS → External resolution required
